# HERMES — Merge LoRA and LiteRT-LM export preparation
This notebook merges adapters into a Hugging Face checkpoint. LiteRT-LM conversion is a separate compatibility-sensitive step; it is NOT claimed complete until an actual `.litertlm` file is produced and validated on-device.

In [ ]:
!pip -q install -U transformers peft accelerate safetensors

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import os
base='Qwen/Qwen2.5-1.5B-Instruct'
adapter='/content/hermes-lora'
assert os.path.isdir(adapter), 'Upload/copy trained adapter to /content/hermes-lora first'
model=AutoModelForCausalLM.from_pretrained(base,device_map='cpu',torch_dtype='auto',low_cpu_mem_usage=True)
model=PeftModel.from_pretrained(model,adapter).merge_and_unload()
out='/content/hermes-merged'
model.save_pretrained(out,safe_serialization=True,max_shard_size='2GB')
AutoTokenizer.from_pretrained(base).save_pretrained(out)
print('Merged checkpoint:',out)

## LiteRT-LM conversion (manual compatibility gate)
1. Check the current [Google AI Edge PyTorch conversion guide](https://ai.google.dev/edge/litert/conversion/pytorch) and [LiteRT-LM repository](https://github.com/google-ai-edge/LiteRT-LM) for Qwen2.5 exporter support and exact current commands.
2. Set up the supported exporter environment; point it to `/content/hermes-merged` and choose a supported quantization mode.
3. Produce `.litertlm` and verify that the file is non-empty and recognized by the installed HERMES version.
4. Run inference on the Xiaomi device and compare against held-out commands. Do not treat a merged checkpoint as a LiteRT-LM file.

Exporter commands are deliberately not fabricated because the tooling and supported targets change.